# MoE Scaling: DeepSeek V3 671B on Ascend 910C

Explore EP/TP trade-offs for large MoE models. Key questions:
- How does Expert Parallelism (EP) affect epoch time and memory?
- What's the optimal EP × TP × PP combination for DSV3 671B?
- How does the model scale across different cluster sizes?

In [ ]:
from llm_perf.config import load_model_config, load_hardware_config, WorkloadConfig, ParallelismConfig
from llm_perf.model import LLMPerformanceModel
from llm_perf.report import format_table

## 1. Load DeepSeek V3 671B

In [ ]:
model = load_model_config("../configs/models/deepseekv3_671b.yaml")
hw = load_hardware_config("../configs/hardware/ascend_910c.yaml")

layer = model.get_layers()[0]
print(f"Model: {model.name}")
print(f"  Layers: {model.num_layers}, Hidden: {model.hidden_size}")
print(f"  Attention: {layer.attention} (heads={layer.num_heads}, d_c={layer.kv_compression_dim})")
print(f"  FFN: {layer.ffn} ({layer.num_experts} experts, top-{layer.top_k}, +{layer.num_shared_experts} shared)")
print(f"  Residual: {layer.residual} (expansion={layer.mhc_expansion})")

perf = LLMPerformanceModel(model, hw)

rl_cfg = WorkloadConfig(
    total_prompts=10000, group_size=8,
    avg_prompt_len=512, avg_response_len=2048,
    max_response_len=4096,
    train_micro_batch_size=2, gradient_accumulation_steps=8,
    gen_batch_size=32,
)

## 2. EP Sweep: Fixed Cluster (1024 devices)
How does EP affect performance? More EP = less expert memory per device, but more AllToAll communication.

In [ ]:
total_devices = 1024
tp = 8
pp = 4

print(f"Cluster: {total_devices} devices, TP={tp}, PP={pp}")
print(f"{'EP':>4} {'DP':>4} {'Epoch(h)':>10} {'Bottleneck':<12} {'Gen TPS':>10} {'Train TPS':>10} {'Train Mem':>10} {'Status':>8}")
print("-" * 75)

for ep in [1, 2, 4, 8, 16, 32]:
    dp = total_devices // (tp * pp * ep)
    if dp < 1:
        continue
    gen_p = ParallelismConfig(tp=tp, dp=total_devices // tp)
    train_p = ParallelismConfig(tp=tp, pp=pp, dp=dp, ep=ep)
    
    r = perf.derive_targets(total_devices, rl_cfg, gen_p, train_p, time_budget_hours=24)
    status = "OK" if r.memory.train_feasible else "OOM"
    print(f"{ep:>4} {dp:>4} {r.epoch_time_hours:>10.2f} {r.bottleneck:<12} "
          f"{r.gen_tps_target:>10,.0f} {r.train_tps_target:>10,.0f} "
          f"{r.memory.total_train_gb:>8.1f}GB {status:>8}")

## 3. Cluster Size Scaling
Fix EP=8, TP=8, PP=4. Scale from 256 to 2048 devices — how does epoch time scale?

In [ ]:
ep, tp, pp = 8, 8, 4

print(f"Fixed: TP={tp}, PP={pp}, EP={ep}")
print(f"{'Devices':>8} {'DP':>4} {'Epoch(h)':>10} {'Gen TPS':>10} {'Train TPS':>10} {'Speedup':>8}")
print("-" * 58)

base_time = None
for n_devices in [256, 512, 1024, 2048]:
    dp = n_devices // (tp * pp * ep)
    if dp < 1:
        continue
    gen_p = ParallelismConfig(tp=tp, dp=n_devices // tp)
    train_p = ParallelismConfig(tp=tp, pp=pp, dp=dp, ep=ep)
    
    r = perf.derive_targets(n_devices, rl_cfg, gen_p, train_p)
    if base_time is None:
        base_time = r.epoch_time_hours
    speedup = base_time / r.epoch_time_hours if r.epoch_time_hours > 0 else 0
    print(f"{n_devices:>8} {dp:>4} {r.epoch_time_hours:>10.2f} "
          f"{r.gen_tps_target:>10,.0f} {r.train_tps_target:>10,.0f} {speedup:>7.1f}x")

## 4. TP × EP Trade-off
With 1024 devices and PP=4, explore the TP×EP design space.

In [ ]:
total_devices = 1024
pp = 4

print(f"Cluster: {total_devices} devices, PP={pp}")
print(f"{'TP':>4} {'EP':>4} {'DP':>4} {'Epoch(h)':>10} {'Train Mem':>10} {'Status':>8}")
print("-" * 50)

for tp in [4, 8]:
    for ep in [4, 8, 16, 32]:
        dp = total_devices // (tp * pp * ep)
        if dp < 1:
            continue
        gen_p = ParallelismConfig(tp=tp, dp=total_devices // tp)
        train_p = ParallelismConfig(tp=tp, pp=pp, dp=dp, ep=ep)
        
        r = perf.derive_targets(total_devices, rl_cfg, gen_p, train_p)
        status = "OK" if r.memory.train_feasible else "OOM"
        print(f"{tp:>4} {ep:>4} {dp:>4} {r.epoch_time_hours:>10.2f} "
              f"{r.memory.total_train_gb:>8.1f}GB {status:>8}")

## 5. Full Report: Best Configuration

In [ ]:
# Pick the best config from above analysis and show full report
gen_p = ParallelismConfig(tp=8, dp=128)
train_p = ParallelismConfig(tp=8, pp=4, dp=4, ep=8)

r = perf.derive_targets(1024, rl_cfg, gen_p, train_p, time_budget_hours=24)
print(format_table(r))